# 1. Setup

## Libraries and imports

In [103]:
!pip install pandas


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [104]:
import pandas as pd
import numpy as np

## Loading NHASES datasets

In [105]:
# DEMOGRAPHIC DATA
demo = pd.read_sas("../data/DEMO_L.xpt", format="xport")

demo

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11928,142306.0,12.0,2.0,1.0,9.0,NaN,2.0,2.0,1.0,111.0,...,1.0,3.0,3.0,3.0,NaN,11147.192563,13459.129019,176.0,1.0,2.01
11929,142307.0,12.0,2.0,2.0,49.0,NaN,4.0,4.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,69419.620456,64962.328962,181.0,1.0,NaN
11930,142308.0,12.0,2.0,1.0,50.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,32696.313477,44367.534132,183.0,2.0,1.95
11931,142309.0,12.0,2.0,1.0,40.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,30547.974564,46249.361849,176.0,1.0,3.11


In [106]:
# EXAMINATION DATA
bmx = pd.read_sas("../data/BMX_L.xpt", format="xport") # body measures
bpxo = pd.read_sas("../data/BPXO_L.xpt", format="xport") # measured blood pressure

In [107]:
# LABORATORY DATA
ghb = pd.read_sas("../data/GHB_L.xpt", format="xport") # glycohemoglobin (HbA1c)
glu = pd.read_sas("../data/GLU_L.xpt", format="xport") # fasting glucose (optional validation)

In [108]:
# QUESTIONNAIRE DATA
paq = pd.read_sas("../data/PAQ_L.xpt", format="xport") # physical activity and sedentary behavior
smq = pd.read_sas("../data/SMQ_L.xpt", format="xport")
bpq = pd.read_sas("../data/BPQ_L.xpt", format="xport")
diq = pd.read_sas("../data/DIQ_L.xpt", format="xport")

## Setting up audit log

In [109]:
audit_log = []

In [110]:
def add_audit_finding(
    dataset,
    column,
    issue,
    example=None,
    action_stage=None,
    planned_action=None
):
    finding = {
        "dataset": dataset,
        "column": column,
        "issue": issue,
        "example": example,
        "action_stage": action_stage,
        "planned_action": planned_action
    }

    audit_log.append(finding)

# 2. Retaining and Renaming Required Columns

## Demographic Data

In [111]:
demo.columns

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN',
       'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4',
       'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND',
       'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR',
       'SDMVSTRA', 'SDMVPSU', 'INDFMPIR'],
      dtype='str')

In [112]:
demo_req_columns = demo[[
    'SEQN', 'RIDAGEYR', 'RIAGENDR', 
    'RIDRETH3', 'DMDEDUC2', 'INDFMPIR', 
    'RIDEXPRG', 'WTMEC2YR', 'SDMVSTRA', 
    'SDMVPSU'
    ]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'RIDAGEYR' : 'age_years', 
        'RIAGENDR' : 'sex', 
        'RIDRETH3' : 'race_ethnicity', 
        'DMDEDUC2': 'education_level', 
        'INDFMPIR' : 'income_poverty_ratio', 
        'RIDEXPRG' : 'pregnancy_status', 
        'WTMEC2YR' : 'mec_exam_weight', 
        'SDMVSTRA' : 'survey_stratum', 
        'SDMVPSU' : 'survey_psu'
    }
    )

demo_req_columns

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
3,130381.0,5.0,2.0,7.0,NaN,1.53,NaN,55698.607106,182.0,2.0
4,130382.0,2.0,1.0,3.0,NaN,3.60,NaN,36434.146346,182.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11928,142306.0,9.0,1.0,2.0,NaN,2.01,NaN,13459.129019,176.0,1.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,64962.328962,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,44367.534132,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,46249.361849,176.0,1.0


## Examination Data

In [113]:
bmx.columns
bmx_req_columns = bmx[[
    'SEQN', 'BMXBMI', 'BMXWAIST'
]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'BMXBMI' : 'bmi', 
        'BMXWAIST' : 'waist_cm'
    }
)

In [114]:
bpxo.columns
bpxo_req_columns = bpxo[[
    'SEQN', 'BPXOSY1', 'BPXOSY2', 
    'BPXOSY3', 'BPXODI1', 'BPXODI2', 
    'BPXODI3'
]].rename(
    columns = {
        'SEQN' : 'participant_id', 
        'BPXOSY1' : 'systolic_1', 
        'BPXOSY2' : 'systolic_2',
        'BPXOSY3' : 'systolic_3',
        'BPXODI1' : 'diastolic_1',
        'BPXODI2' : 'diastolic_2',
        'BPXODI3' : 'diastolic_3'
    }
)

# derived : mean_systolic_bp, mean_diastolic_bp

## Laboratory Data

In [115]:
ghb.columns

ghb_req_columns = ghb.rename(columns = {
    'SEQN' : 'participant_id', 
    'LBXGH' : 'hba1c_pct',    
    'WTPH2YR' : 'phlebotomy_weight', 

})

In [116]:
glu_req_columns = glu[[
    'SEQN', 'LBXGLU', 'WTSAF2YR'
]].rename( columns = {
    'SEQN' : 'participant_id', 
    'LBXGLU' : 'fasting_glucose_mg_dl', 
    'WTSAF2YR' : 'fasting_subsample_weight'
})

glu_req_columns

,participant_id,fasting_glucose_mg_dl,fasting_subsample_weight
0,130378.0,113.0,1.200253e+05
1,130379.0,99.0,5.397605e-79
2,130380.0,156.0,1.450908e+05
3,130386.0,100.0,8.259962e+04
4,130394.0,88.0,1.004203e+05
...,...,...,...
3991,142301.0,110.0,3.112337e+04
3992,142303.0,160.0,1.095823e+05
3993,142305.0,132.0,8.479001e+04
3994,142308.0,NaN,5.397605e-79


## Questionnaire Data

In [117]:
paq_req_columns = paq.rename(columns = {
    'SEQN' : 'participant_id', 
    'PAD790Q' : 'moderate_pa_frequency', 
    'PAD790U' : 'moderate_pa_unit', 
    'PAD800' : 'moderate_pa_minutes_session', 
    'PAD810Q' : 'vigorous_pa_frequency', 
    'PAD810U' : 'vigorous_pa_unit', 
    'PAD820' : 'vigorous_pa_minutes_session',
    'PAD680' : 'sedentary_min_day'
})

In [118]:
smq_req_columns = smq[['SEQN', 'SMQ020', 'SMQ040']].rename(columns = {
    'SEQN' : 'participant_id', 
    'SMQ020' : 'ever_100_cigarettes', 
    'SMQ040' : 'current_smoking_frequency', 
})

In [119]:
bpq_req_columns = bpq[['SEQN', 'BPQ020', 'BPQ150', 'BPQ080', 'BPQ101D']].rename(columns = {
    'SEQN' : 'participant_id', 
    'BPQ020' : 'hypertension_history', 
    'BPQ150' : 'bp_medication', 
    'BPQ080' : 'high_cholesterol_history', 
    'BPQ101D' : 'cholesterol_medication'
})

In [120]:
diq_req_columns = diq[[
    'SEQN', 'DIQ010', 'DIQ160'
]].rename(columns = {
    'SEQN' : 'participant_id', 
    'DIQ010' : 'diagnosed_diabetes', 
    'DIQ160' : 'prediabetes_history', 
})

### Audit

In [121]:
# paq 
# moderate_pa_unit, vigorous_pa_unit
# python read as bytes object

# smq
# smoking_status 
# to be derived

# 3. Establishing Cohort
Cohort represents adults ages 20+ and excludes women that are pregnant or cannot ascertain pregnancy status

In [122]:
# normalizing education_level so 7 and 9 are treated as NaN (missing value)
# Adults 20+: 
# 1 <9th grade; 
# 2 9-11th; 
# 3 HS/GED; 
# 4 some college/AA; 
# 5 college+. 
# Codes 7/9 -> missing.

demo_req_columns['education_level'].unique()

array([ 5.,  3., nan,  2.,  4.,  1.,  9.])

In [123]:
# creating adult cohort 
# age_years >= 20

demo_adults = demo_req_columns[
    demo_req_columns['age_years'] >= 20
].copy()

demo_adults

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,5.437446e+04,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,3.408472e+04,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,8.119628e+04,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,NaN,5.397605e-79,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,NaN,5.397605e-79,187.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,NaN,4.348341e+04,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,6.496233e+04,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,4.436753e+04,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,4.624936e+04,176.0,1.0


In [124]:
demo_adults['education_level'].isna().mean()*100

# missing education level also have a lot of missing income poverty ratio

np.float64(0.1920860545524395)

In [125]:
# checking pregnancy status
# For females 20-44: 
# 1 pregnant; 
# 2 not pregnant; 
# 3 cannot ascertain.

# there are 41 women with a pregnancy status of 1
# there are 397 women with a pregnancy status of 3

demo_adults['pregnancy_status'].value_counts(dropna=False)

pregnancy_status
NaN    6306
2.0    1065
3.0     397
1.0      41
Name: count, dtype: int64

In [126]:
# dropping pregnancy status 1 and pregnancy status 3

demo_cohort = demo_adults[
    ~(
        (demo_adults["pregnancy_status"] == 1) |
        ((demo_adults["pregnancy_status"] == 3) & (demo_adults["sex"] == 2))
    )
].copy()

demo_cohort['pregnancy_status'].value_counts(dropna=False)

demo_cohort

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,pregnancy_status,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,NaN,5.437446e+04,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,NaN,3.408472e+04,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,8.119628e+04,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,NaN,5.397605e-79,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,NaN,5.397605e-79,187.0,2.0
...,...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,NaN,4.348341e+04,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,NaN,6.496233e+04,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,NaN,4.436753e+04,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,NaN,4.624936e+04,176.0,1.0


In [127]:
# making sure there are no NaN values for females between 20 and 44

pregnancy_nan_check = demo_cohort[
    (demo_cohort["sex"] == 2) &
    (demo_cohort["age_years"].between(20, 44)) &
    (demo_cohort["pregnancy_status"].isna())
]

pregnancy_nan_check[
    ["participant_id", "age_years", "sex", "pregnancy_status"]
]

,participant_id,age_years,sex,pregnancy_status


In [128]:
# we can drop pregnancy column now since it was only useful as a cohort filter

demo_cohort = demo_cohort.drop(columns=["pregnancy_status"])
demo_cohort

# demo cohort now only consists of adults >= 20 years, and candidates that are not pregnant or cannot ascertain if they are pregnant

,participant_id,age_years,sex,race_ethnicity,education_level,income_poverty_ratio,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,5.437446e+04,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,3.408472e+04,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,8.119628e+04,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,5.397605e-79,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,5.397605e-79,187.0,2.0
...,...,...,...,...,...,...,...,...,...
11927,142305.0,76.0,2.0,1.0,1.0,2.25,4.348341e+04,180.0,2.0
11929,142307.0,49.0,2.0,4.0,5.0,NaN,6.496233e+04,181.0,1.0
11930,142308.0,50.0,1.0,2.0,4.0,1.95,4.436753e+04,183.0,2.0
11931,142309.0,40.0,1.0,2.0,4.0,3.11,4.624936e+04,176.0,1.0


In [129]:
datasets = {
    "demo": demo_cohort,
    "bmx": bmx_req_columns,
    "paq": paq_req_columns,
    "smq": smq_req_columns,
    "bpxo": bpxo_req_columns,
    "bpq": bpq_req_columns,
    "diq": diq_req_columns,
    "ghb": ghb_req_columns,
    "glu": glu_req_columns
}

### Audit

In [130]:
add_audit_finding(
    dataset="DEMO",
    column="education_level",
    issue="NHANES special response codes 7 and/or 9 are present in education level.",
    example="Codes represent Refused/Don't Know responses rather than valid education categories.",
    action_stage="cleaning",
    planned_action="Recode special response codes 7 and 9 to missing (NaN)."
)

# 4. Auditing Data

## Component coverage audit

Of the 7,371 eligible adults, how many actually have a record in each NHANES component?

In [131]:
cohort_ids = demo_cohort["participant_id"]

coverage_results = []

for name, df in datasets.items():

    available = cohort_ids.isin(df["participant_id"])

    coverage_results.append({
        "dataset": name.upper(),
        "cohort_size": len(demo_cohort),
        "participants_available": available.sum(),
        "participants_not_available": (~available).sum(),
        "coverage_percent": round(available.mean() * 100, 2)
    })

coverage_summary = pd.DataFrame(coverage_results)

coverage_summary

,dataset,cohort_size,participants_available,participants_not_available,coverage_percent
0,DEMO,7371,7371,0,100.00
1,BMX,7371,5995,1376,81.33
2,PAQ,7371,7371,0,100.00
3,SMQ,7371,7371,0,100.00
4,BPXO,7371,5995,1376,81.33
5,BPQ,7371,7371,0,100.00
6,DIQ,7371,7371,0,100.00
7,GHB,7371,5995,1376,81.33
8,GLU,7371,3382,3989,45.88


In [132]:
coverage_summary[
    coverage_summary["coverage_percent"] < 100
].sort_values("coverage_percent")

,dataset,cohort_size,participants_available,participants_not_available,coverage_percent
8,GLU,7371,3382,3989,45.88
1,BMX,7371,5995,1376,81.33
4,BPXO,7371,5995,1376,81.33
7,GHB,7371,5995,1376,81.33


### Audit

In [133]:
add_audit_finding(
    dataset="BMX/BPXO/GHB",
    column="component_record",
    issue=(
        "Only 5,995 of 7,371 cohort participants (81.33%) have records "
        "in these components; 1,376 participants have no component record."
    ),
    example="BMX, BPXO, and GHB have the same cohort coverage pattern.",
    action_stage="modeling",
    planned_action=(
        "Treat absence of the entire component separately from variable-level "
        "missingness. Do not impute participants who have no component record."
    )
)

add_audit_finding(
    dataset="GLU",
    column="component_record",
    issue=(
        "Only 3,382 of 7,371 cohort participants (45.88%) have a fasting "
        "glucose component record."
    ),
    example="3,989 cohort participants have no GLU record.",
    action_stage="modeling",
    planned_action=(
        "Treat GLU as a fasting subsample rather than assuming the missing "
        "participants have ordinary missing values. Use the appropriate "
        "fasting subsample weight for survey analyses."
    )
)

## Source level Missingness
Among the people who appear in this original NHANES file, how much of this variable is missing?

In [134]:
# checking for missingness and duplicates

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Unique participants:", df["participant_id"].nunique())
    print("Duplicate IDs:", df["participant_id"].duplicated().sum())
    print("Missing IDs:", df["participant_id"].isna().sum())
    print(df.isna().sum())


DEMO
Shape: (7371, 9)
Unique participants: 7371
Duplicate IDs: 0
Missing IDs: 0
participant_id             0
age_years                  0
sex                        0
race_ethnicity             0
education_level           10
income_poverty_ratio    1210
mec_exam_weight            0
survey_stratum             0
survey_psu                 0
dtype: int64

BMX
Shape: (8860, 3)
Unique participants: 8860
Duplicate IDs: 0
Missing IDs: 0
participant_id      0
bmi               389
waist_cm          670
dtype: int64

PAQ
Shape: (8153, 8)
Unique participants: 8153
Duplicate IDs: 0
Missing IDs: 0
participant_id                    0
moderate_pa_frequency            18
moderate_pa_unit                  0
moderate_pa_minutes_session    1763
vigorous_pa_frequency            14
vigorous_pa_unit                  0
vigorous_pa_minutes_session    4466
sedentary_min_day                15
dtype: int64

SMQ
Shape: (9015, 3)
Unique participants: 9015
Duplicate IDs: 0
Missing IDs: 0
participant_id           

In [135]:
# evaluating missingness percentage

for name, df in datasets.items():
    print(f"\n{name.upper()}")

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    print(missing)


DEMO
                      missing_count  missing_percent
participant_id                    0             0.00
age_years                         0             0.00
sex                               0             0.00
race_ethnicity                    0             0.00
education_level                  10             0.14
income_poverty_ratio           1210            16.42
mec_exam_weight                   0             0.00
survey_stratum                    0             0.00
survey_psu                        0             0.00

BMX
                missing_count  missing_percent
participant_id              0             0.00
bmi                       389             4.39
waist_cm                  670             7.56

PAQ
                             missing_count  missing_percent
participant_id                           0             0.00
moderate_pa_frequency                   18             0.22
moderate_pa_unit                         0             0.00
moderate_pa_minutes_sessio

### Audit

In [136]:
add_audit_finding(
    dataset="ALL",
    column="participant_id",
    issue=(
        "No missing participant IDs or duplicate participant IDs were detected "
        "in the retained source datasets."
    ),
    example="All component datasets passed participant ID uniqueness checks.",
    action_stage="merge",
    planned_action=(
        "No ID-level cleaning required before one-to-one cohort merges."
    )
)

## Cohort level missingness
Among the people who appear in the cohort, how many do not have this variable available?

Merging on participant_id to understand missigness in case of a merge.

In [137]:
cohort_datasets = {
    "demo": demo_cohort
}

for name, df in datasets.items():

    if name == "demo":
        continue

    cohort_datasets[name] = (
        demo_cohort[["participant_id"]]
        .merge(
            df,
            on="participant_id",
            how="left",
            validate="one_to_one"
        )
    )

In [138]:
for name, df in cohort_datasets.items():

    print(f"\n{'=' * 50}")
    print(name.upper())
    print("=" * 50)

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    print(missing)


DEMO
                      missing_count  missing_percent
participant_id                    0             0.00
age_years                         0             0.00
sex                               0             0.00
race_ethnicity                    0             0.00
education_level                  10             0.14
income_poverty_ratio           1210            16.42
mec_exam_weight                   0             0.00
survey_stratum                    0             0.00
survey_psu                        0             0.00

BMX
                missing_count  missing_percent
participant_id              0             0.00
bmi                      1463            19.85
waist_cm                 1665            22.59

PAQ
                             missing_count  missing_percent
participant_id                           0             0.00
moderate_pa_frequency                    8             0.11
moderate_pa_unit                         0             0.00
moderate_pa_minutes_sessio

In [139]:
missing_reason_results = []

for name, df in datasets.items():

    if name == "demo":
        continue

    source_ids = set(df["participant_id"])

    cohort_df = cohort_datasets[name]

    for column in df.columns:

        if column == "participant_id":
            continue

        no_component_record = (
            ~cohort_df["participant_id"].isin(source_ids)
        ).sum()

        record_exists_but_value_missing = (
            cohort_df["participant_id"].isin(source_ids)
            & cohort_df[column].isna()
        ).sum()

        missing_reason_results.append({
            "dataset": name.upper(),
            "column": column,
            "no_component_record": no_component_record,
            "record_exists_but_value_missing": record_exists_but_value_missing,
            "total_missing": cohort_df[column].isna().sum()
        })

missing_reason_summary = pd.DataFrame(missing_reason_results)

missing_reason_summary

,dataset,column,no_component_record,record_exists_but_value_missing,total_missing
0,BMX,bmi,1376,87,1463
1,BMX,waist_cm,1376,289,1665
2,PAQ,moderate_pa_frequency,0,8,8
3,PAQ,moderate_pa_unit,0,0,0
4,PAQ,moderate_pa_minutes_session,0,1623,1623
5,PAQ,vigorous_pa_frequency,0,8,8
6,PAQ,vigorous_pa_unit,0,0,0
7,PAQ,vigorous_pa_minutes_session,0,4138,4138
8,PAQ,sedentary_min_day,0,8,8
9,SMQ,ever_100_cigarettes,0,10,10


### Audit

In [140]:
add_audit_finding(
    dataset="DEMO",
    column="income_poverty_ratio",
    issue="1,210 cohort participants (16.42%) are missing income-poverty ratio.",
    example="Cohort-level missingness = 16.42%.",
    action_stage="modeling",
    planned_action=(
        "Preserve as missing during cleaning. Decide on imputation, missing "
        "indicator, or exclusion during modeling/preprocessing."
    )
)

add_audit_finding(
    dataset="BMX",
    column="bmi / waist_cm",
    issue=(
        "BMI is missing for 1,463 participants (19.85%) and waist circumference "
        "for 1,665 (22.59%). Most missingness comes from 1,376 participants "
        "with no BMX component record; an additional 87 BMI and 289 waist "
        "values are missing among participants with a BMX record."
    ),
    example="Component noncoverage and within-component missingness are both present.",
    action_stage="modeling",
    planned_action=(
        "Keep component noncoverage separate from within-component missingness "
        "when defining the analytic sample and handling missing predictors."
    )
)

add_audit_finding(
    dataset="BPXO",
    column="systolic_1-3 / diastolic_1-3",
    issue=(
        "Blood-pressure measurements have approximately 21.3-21.5% cohort-level "
        "missingness. 1,376 participants have no BPXO record, with an additional "
        "191-210 missing measurements depending on reading number."
    ),
    example="Missingness increases slightly across repeated BP readings.",
    action_stage="feature_engineering",
    planned_action=(
        "Preserve individual readings during cleaning. After cleaning, derive "
        "mean systolic and mean diastolic blood pressure using available valid readings."
    )
)

add_audit_finding(
    dataset="GHB",
    column="hba1c_pct",
    issue=(
        "HbA1c is missing for 1,662 cohort participants (22.55%). "
        "1,376 have no GHB component record and 286 have a GHB record "
        "but no HbA1c value."
    ),
    example="5,709 cohort participants have an HbA1c measurement.",
    action_stage="target_definition",
    planned_action=(
        "Preserve missing HbA1c values. Define outcome eligibility after cleaning "
        "and target construction rather than imputing HbA1c."
    )
)

add_audit_finding(
    dataset="GLU",
    column="fasting_glucose_mg_dl",
    issue=(
        "Fasting glucose is missing for 4,197 cohort participants (56.94%). "
        "3,989 have no GLU component record and 208 have a GLU record "
        "but no fasting glucose result."
    ),
    example="GLU is a substantially smaller fasting subsample.",
    action_stage="modeling",
    planned_action=(
        "Do not impute fasting glucose across the full cohort. Use only where "
        "appropriate for secondary or subsample analyses."
    )
)

## Audit questionnaire skip logic

In [141]:
# smoking skip pattern
pd.crosstab(
    cohort_datasets["smq"]["ever_100_cigarettes"],
    cohort_datasets["smq"]["current_smoking_frequency"].isna(),
    margins=True
)

current_smoking_frequency,False,True,All
ever_100_cigarettes,,,
1.0,3124,0,3124
2.0,0,4224,4224
7.0,0,7,7
9.0,0,6,6
All,3124,4237,7361


In [142]:
# hypertension medication skip pattern
pd.crosstab(
    cohort_datasets["bpq"]["hypertension_history"],
    cohort_datasets["bpq"]["bp_medication"].isna(),
    margins=True
)

bp_medication,False,True,All
hypertension_history,,,
1.0,2881,0,2881
2.0,0,4477,4477
7.0,0,1,1
9.0,0,10,10
All,2881,4488,7369


In [143]:
# cholesterol medication skip pattern
pd.crosstab(
    cohort_datasets["bpq"]["high_cholesterol_history"],
    cohort_datasets["bpq"]["cholesterol_medication"].isna(),
    margins=True
)

cholesterol_medication,False,All
high_cholesterol_history,,
1.0,3000,3000
2.0,4319,4319
9.0,50,50
All,7369,7369


In [144]:
# prediabetes question vs diabetes diagnosis
pd.crosstab(
    cohort_datasets["diq"]["diagnosed_diabetes"],
    cohort_datasets["diq"]["prediabetes_history"].isna(),
    margins=True
)

prediabetes_history,False,True,All
diagnosed_diabetes,,,
1.0,0,1059,1059
2.0,6043,2,6045
3.0,0,264,264
9.0,1,0,1
All,6044,1325,7369


In [145]:
xpt_zero = 5.397605346934028e-79


paq_audit = cohort_datasets["paq"].copy()

paq_audit["moderate_pa_frequency_audit"] = (
    paq_audit["moderate_pa_frequency"]
    .replace(xpt_zero, 0)
)

pd.crosstab(
    paq_audit["moderate_pa_frequency_audit"] == 0,
    paq_audit["moderate_pa_minutes_session"].isna(),
    margins=True
)

moderate_pa_minutes_session,False,True,All
moderate_pa_frequency_audit,,,
False,5748,52,5800
True,0,1571,1571
All,5748,1623,7371


In [146]:
paq_audit["vigorous_pa_frequency_audit"] = (
    paq_audit["vigorous_pa_frequency"]
    .replace(xpt_zero, 0)
)

pd.crosstab(
    paq_audit["vigorous_pa_frequency_audit"] == 0,
    paq_audit["vigorous_pa_minutes_session"].isna(),
    margins=True
)

vigorous_pa_minutes_session,False,True,All
vigorous_pa_frequency_audit,,,
False,3233,44,3277
True,0,4094,4094
All,3233,4138,7371


### Audit

In [147]:
add_audit_finding(
    dataset="SMQ",
    column="current_smoking_frequency",
    issue=(
        "Missing current smoking frequency is primarily structural. "
        "All 4,224 participants who reported not smoking at least 100 cigarettes "
        "have current_smoking_frequency missing."
    ),
    example=(
        "ever_100_cigarettes = 2: 4,224 participants, all with "
        "current_smoking_frequency missing."
    ),
    action_stage="cleaning",
    planned_action=(
        "Do not treat these values as ordinary missing data. Use "
        "ever_100_cigarettes when deriving smoking status and preserve "
        "Refused/Don't Know responses as missing/unknown."
    )
)

add_audit_finding(
    dataset="BPQ",
    column="bp_medication",
    issue=(
        "Missing BP medication responses are primarily structural. "
        "All 4,477 participants reporting no hypertension history have "
        "bp_medication missing."
    ),
    example=(
        "hypertension_history = 2: 4,477 participants, all with "
        "bp_medication missing."
    ),
    action_stage="cleaning",
    planned_action=(
        "Treat BP medication as not applicable when hypertension history "
        "indicates no hypertension rather than imputing these values."
    )
)

add_audit_finding(
    dataset="DIQ",
    column="prediabetes_history",
    issue=(
        "Prediabetes history missingness is largely structural based on "
        "diabetes diagnosis. All participants reporting diagnosed diabetes "
        "or the code 3 response have prediabetes_history missing."
    ),
    example=(
        "diagnosed_diabetes = 1: 1,059 participants, all missing prediabetes_history; "
        "diagnosed_diabetes = 3: 264 participants, all missing."
    ),
    action_stage="cleaning",
    planned_action=(
        "Interpret prediabetes_history together with diagnosed_diabetes rather "
        "than treating structurally skipped responses as ordinary missing data."
    )
)

add_audit_finding(
    dataset="PAQ",
    column="moderate_pa_minutes_session",
    issue=(
        "Moderate activity session minutes show structural missingness when "
        "moderate activity frequency is zero."
    ),
    example=(
        "All 1,571 participants with moderate_pa_frequency = 0 have "
        "moderate_pa_minutes_session missing; an additional 52 participants "
        "with nonzero frequency also have minutes missing."
    ),
    action_stage="cleaning",
    planned_action=(
        "Treat missing session minutes as not applicable when activity frequency "
        "is zero. Keep the additional missing values among participants with "
        "nonzero frequency as true missing values requiring separate handling."
    )
)

add_audit_finding(
    dataset="PAQ",
    column="vigorous_pa_minutes_session",
    issue=(
        "Vigorous activity session minutes show structural missingness when "
        "vigorous activity frequency is zero."
    ),
    example=(
        "Participants with vigorous_pa_frequency = 0 have "
        "vigorous_pa_minutes_session missing."
    ),
    action_stage="cleaning",
    planned_action=(
        "Treat missing session minutes as not applicable when vigorous activity "
        "frequency is zero. Preserve missing minutes among participants with "
        "nonzero frequency as true missing values."
    )
)

## Understanding categorical value distributions

In [148]:
categorical_columns = {
    "demo": [
        "sex",
        "race_ethnicity",
        "education_level"
    ],

    "paq": [
        "moderate_pa_unit",
        "vigorous_pa_unit"
    ],

    "smq": [
        "ever_100_cigarettes",
        "current_smoking_frequency"
    ],

    "bpq": [
        "hypertension_history",
        "bp_medication",
        "high_cholesterol_history",
        "cholesterol_medication"
    ],

    "diq": [
        "diagnosed_diabetes",
        "prediabetes_history"
    ]
}

for name, columns in categorical_columns.items():

    print(f"\n{'=' * 40}")
    print(name.upper())
    print("=" * 40)

    for column in columns:
        print(f"\n")
        print(datasets[name][column].value_counts(dropna=False))


DEMO


sex
2.0    3883
1.0    3488
Name: count, dtype: int64


race_ethnicity
3.0    4360
4.0     902
2.0     728
1.0     500
7.0     475
6.0     406
Name: count, dtype: int64


education_level
5.0    2488
4.0    2221
3.0    1652
2.0     626
1.0     363
9.0      11
NaN      10
Name: count, dtype: int64

PAQ


moderate_pa_unit
b'W'    4851
b''     1763
b'D'     985
b'M'     472
b'Y'      82
Name: count, dtype: int64


vigorous_pa_unit
b''     4467
b'W'    2590
b'M'     711
b'D'     235
b'Y'     150
Name: count, dtype: int64

SMQ


ever_100_cigarettes
2.0    4878
1.0    3243
NaN     880
9.0       7
7.0       7
Name: count, dtype: int64


current_smoking_frequency
NaN    5772
3.0    2053
1.0     952
2.0     238
Name: count, dtype: int64

BPQ


hypertension_history
2.0    5518
1.0    2969
9.0      10
NaN       3
7.0       1
Name: count, dtype: int64


bp_medication
NaN    5532
1.0    2442
2.0     523
9.0       4
Name: count, dtype: int64


high_cholesterol_history
2.0    5348
1.0    3096


### Audit

In [149]:
add_audit_finding(
    dataset="PAQ",
    column="moderate_pa_unit / vigorous_pa_unit",
    issue=(
        "Physical-activity unit fields were imported from the XPT files as byte "
        "objects and include blank byte values."
    ),
    example="Values include b'D', b'W', b'M', b'Y', and b''.",
    action_stage="cleaning",
    planned_action=(
        "Decode/map byte values to readable categories (Day, Week, Month, Year) "
        "and convert b'' to missing."
    )
)

add_audit_finding(
    dataset="DIQ",
    column="diagnosed_diabetes",
    issue=(
        "The diagnosed diabetes variable contains code 3 in addition to "
        "yes/no responses."
    ),
    example="Code 3 occurs in the source data and represents a distinct response category.",
    action_stage="target_definition",
    planned_action=(
        "Preserve code 3 during cleaning and explicitly decide how it contributes "
        "to diabetes outcome construction rather than automatically treating it as yes/no."
    )
)

## Audit NHANES special response codes

In [150]:
special_codes = {

    "paq": {
        "moderate_pa_frequency": [7777, 9999],
        "moderate_pa_minutes_session": [7777, 9999],
        "vigorous_pa_frequency": [7777, 9999],
        "vigorous_pa_minutes_session": [7777, 9999],
        "sedentary_min_day": [7777, 9999]
    },

    "smq": {
        "ever_100_cigarettes": [7, 9]
    },

    "bpq": {
        "hypertension_history": [7, 9],
        "bp_medication": [7, 9],
        "high_cholesterol_history": [7, 9],
        "cholesterol_medication": [7, 9]
    },

    "diq": {
        "diagnosed_diabetes": [7, 9],
        "prediabetes_history": [7, 9]
    }
}

special_code_findings = []

for dataset_name, columns in special_codes.items():

    df = cohort_datasets[dataset_name]

    for column, codes in columns.items():

        for code in codes:

            count = (df[column] == code).sum()

            if count > 0:
                special_code_findings.append({
                    "dataset": dataset_name.upper(),
                    "column": column,
                    "code": code,
                    "count": count
                })

special_code_findings = pd.DataFrame(special_code_findings)

special_code_findings

,dataset,column,code,count
0,PAQ,moderate_pa_frequency,7777,7
1,PAQ,moderate_pa_frequency,9999,37
2,PAQ,moderate_pa_minutes_session,9999,18
3,PAQ,vigorous_pa_frequency,7777,2
4,PAQ,vigorous_pa_frequency,9999,35
5,PAQ,vigorous_pa_minutes_session,7777,1
6,PAQ,vigorous_pa_minutes_session,9999,10
7,PAQ,sedentary_min_day,7777,5
8,PAQ,sedentary_min_day,9999,66
9,SMQ,ever_100_cigarettes,7,7


### Audit

In [151]:
for dataset_name, group in special_code_findings.groupby("dataset"):

    details = "; ".join(
        f"{row['column']}: {int(row['code'])} ({int(row['count'])} records)"
        for _, row in group.iterrows()
    )

    add_audit_finding(
        dataset=dataset_name,
        column="multiple questionnaire fields",
        issue="NHANES special response codes are present.",
        example=details,
        action_stage="cleaning",
        planned_action=(
            "Recode applicable Refused/Don't Know special-response codes "
            "to missing before feature engineering or statistical analysis."
        )
    )

## Audit continuous values

In [152]:
numeric_columns = {

    "demo": [
        "age_years",
        "income_poverty_ratio"
    ],

    "bmx": [
        "bmi",
        "waist_cm"
    ],

    "paq": [
        "moderate_pa_frequency",
        "moderate_pa_minutes_session",
        "vigorous_pa_frequency",
        "vigorous_pa_minutes_session",
        "sedentary_min_day"
    ],

    "bpxo": [
        "systolic_1",
        "systolic_2",
        "systolic_3",
        "diastolic_1",
        "diastolic_2",
        "diastolic_3"
    ],

    "ghb": [
        "hba1c_pct"
    ],

    "glu": [
        "fasting_glucose_mg_dl"
    ]
}

In [153]:

for name, df in cohort_datasets.items():

    counts = {}

    for column in df.select_dtypes(include="number").columns:

        count = (df[column] == xpt_zero).sum()

        if count > 0:
            counts[column] = count

    if counts:
        print(f"\n{name.upper()}")
        print(counts)


DEMO
{'income_poverty_ratio': np.int64(64), 'mec_exam_weight': np.int64(1376)}

PAQ
{'moderate_pa_frequency': np.int64(1571), 'vigorous_pa_frequency': np.int64(4094), 'sedentary_min_day': np.int64(8)}

GHB
{'phlebotomy_weight': np.int64(261)}

GLU
{'fasting_subsample_weight': np.int64(452)}


In [154]:
# temporary audit copy

for name, columns in numeric_columns.items():

    audit_df = cohort_datasets[name][
        ["participant_id"] + columns
    ].copy()

    # Treat XPT representation of zero as actual zero
    audit_df[columns] = audit_df[columns].replace(
        xpt_zero,
        0
    )

    # Temporarily exclude known NHANES special codes
    # so they don't distort the distribution
    if name in special_codes:

        for column, codes in special_codes[name].items():

            if column in audit_df.columns:

                audit_df[column] = (
                    audit_df[column]
                    .replace(codes, np.nan)
                )

    print(f"\n{'=' * 50}")
    print(name.upper())
    print("=" * 50)

    display(
        audit_df[columns]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.50,
                0.95,
                0.99
            ]
        )
        .T
    )


DEMO


,count,mean,std,min,1%,5%,50%,95%,99%,max
age_years,7371.0,54.889160,17.161332,20.0,21.0,25.00,58.00,80.0,80.0,80.0
income_poverty_ratio,6161.0,2.921417,1.655707,0.0,0.0,0.46,2.83,5.0,5.0,5.0



BMX


,count,mean,std,min,1%,5%,50%,95%,99%,max
bmi,5908.0,29.813761,7.329927,11.1,18.2,20.4,28.5,43.6,52.693,74.8
waist_cm,5706.0,101.080775,16.868652,62.4,70.2,76.4,99.5,131.0,148.190,187.0



PAQ


...

,count,mean,std,min,1%,5%,50%,95%,99%,max
systolic_1,5804.0,123.210544,18.539453,75.0,88.03,98.0,121.0,157.0,180.00,232.0
systolic_2,5797.0,123.065551,18.480937,65.0,88.00,98.0,121.0,157.0,178.00,233.0
systolic_3,5785.0,122.868280,18.375472,62.0,88.00,98.0,121.0,156.0,179.16,232.0
diastolic_1,5804.0,75.355272,11.360566,37.0,52.00,59.0,75.0,95.0,107.00,142.0
diastolic_2,5797.0,74.678972,11.286436,32.0,50.00,58.0,74.0,95.0,106.04,139.0
diastolic_3,5785.0,74.309248,11.193505,26.0,50.00,58.0,74.0,94.0,104.16,136.0



GHB


,count,mean,std,min,1%,5%,50%,95%,99%,max
hba1c_pct,5709.0,5.794377,1.115476,3.2,4.5,4.9,5.5,7.7,10.992,17.1



GLU


,count,mean,std,min,1%,5%,50%,95%,99%,max
fasting_glucose_mg_dl,3174.0,109.983302,34.29373,59.0,78.73,86.0,101.0,165.35,277.27,561.0


In [155]:
for name, columns in numeric_columns.items():

    audit_df = cohort_datasets[name][
        ["participant_id"] + columns
    ].copy()

    audit_df[columns] = audit_df[columns].replace(
        xpt_zero,
        0
    )

    if name in special_codes:

        for column, codes in special_codes[name].items():

            if column in audit_df.columns:
                audit_df[column] = audit_df[column].replace(
                    codes,
                    np.nan
                )

    for column in columns:

        print(f"\n{name.upper()} - {column}")

        print("Lowest values:")
        display(
            audit_df[
                ["participant_id", column]
            ]
            .nsmallest(5, column)
        )

        print("Highest values:")
        display(
            audit_df[
                ["participant_id", column]
            ]
            .nlargest(5, column)
        )


DEMO - age_years
Lowest values:


,participant_id,age_years
196,130574.0,20.0
319,130697.0,20.0
831,131209.0,20.0
870,131248.0,20.0
1279,131657.0,20.0


Highest values:


,participant_id,age_years
30,130408.0,80.0
49,130427.0,80.0
53,130431.0,80.0
134,130512.0,80.0
141,130519.0,80.0



DEMO - income_poverty_ratio
Lowest values:


...

,participant_id,hba1c_pct
1945,133541.0,17.1
4417,137535.0,17.1
4897,138311.0,15.6
3159,135544.0,14.8
1093,132155.0,14.6



GLU - fasting_glucose_mg_dl
Lowest values:


,participant_id,fasting_glucose_mg_dl
4066,137006.0,59.0
6473,140834.0,59.0
1227,132378.0,64.0
312,130890.0,67.0
1733,133183.0,69.0


Highest values:


,participant_id,fasting_glucose_mg_dl
6019,140143.0,561.0
2149,133882.0,460.0
1136,132231.0,417.0
789,131672.0,385.0
2272,134096.0,366.0


### Audit

In [156]:
add_audit_finding(
    dataset="DEMO/PAQ/GHB/GLU",
    column="multiple numeric fields",
    issue=(
        "The SAS XPT representation of zero "
        "(5.397605346934028e-79) appears in multiple numeric variables."
    ),
    example=(
        "DEMO income_poverty_ratio: 64; DEMO mec_exam_weight: 1,376; "
        "PAQ moderate_pa_frequency: 1,571; vigorous_pa_frequency: 4,094; "
        "sedentary_min_day: 8; GHB phlebotomy_weight: 261; "
        "GLU fasting_subsample_weight: 452."
    ),
    action_stage="cleaning",
    planned_action=(
        "Convert the XPT zero representation to numeric 0 before calculating "
        "derived features, interpreting distributions, or using survey weights."
    )
)

## Audit weight values

mec_exam_weight
phlebotomy_weight
fasting_subsample_weight
survey_stratum
survey_psu

In [157]:
weight_columns = {
    "demo": "mec_exam_weight",
    "ghb": "phlebotomy_weight",
    "glu": "fasting_subsample_weight"
}

for name, column in weight_columns.items():

    df = cohort_datasets[name]

    print(f"\n{name.upper()} - {column}")

    print("Missing:", df[column].isna().sum())
    print("Zero:", (df[column] == 0).sum())
    print("Negative:", (df[column] < 0).sum())

    display(
        df[column].describe()
    )


DEMO - mec_exam_weight
Missing: 0
Zero: 0
Negative: 0


count    7.371000e+03
mean     3.277314e+04
std      3.071841e+04
min      5.397605e-79
25%      1.335454e+04
50%      2.611318e+04
75%      4.436910e+04
max      2.271083e+05
Name: mec_exam_weight, dtype: float64


GHB - phlebotomy_weight
Missing: 1376
Zero: 0
Negative: 0


count    5.995000e+03
mean     4.033918e+04
std      3.168328e+04
min      5.397605e-79
25%      1.946919e+04
50%      3.189058e+04
75%      5.091838e+04
max      2.417289e+05
Name: phlebotomy_weight, dtype: float64


GLU - fasting_subsample_weight
Missing: 3989
Zero: 0
Negative: 0


count    3.382000e+03
mean     7.162385e+04
std      6.522894e+04
min      5.397605e-79
25%      3.128419e+04
50%      5.686205e+04
75%      9.497383e+04
max      5.619224e+05
Name: fasting_subsample_weight, dtype: float64

In [158]:
print("Missing survey stratum:")
print(demo_cohort["survey_stratum"].isna().sum())

print("\nMissing survey PSU:")
print(demo_cohort["survey_psu"].isna().sum())

print("\nPSU values:")
print(
    demo_cohort["survey_psu"]
    .value_counts(dropna=False)
)

print("\nNumber of PSUs per stratum:")

psu_by_stratum = (
    demo_cohort
    .groupby("survey_stratum")["survey_psu"]
    .nunique()
)

display(psu_by_stratum)

Missing survey stratum:
0

Missing survey PSU:
0

PSU values:
survey_psu
1.0    3749
2.0    3622
Name: count, dtype: int64

Number of PSUs per stratum:


survey_stratum
173.0    2
174.0    2
175.0    2
176.0    2
177.0    2
178.0    2
179.0    2
180.0    2
181.0    2
182.0    2
183.0    2
184.0    2
185.0    2
186.0    2
187.0    2
Name: survey_psu, dtype: int64

### Audit

In [159]:
add_audit_finding(
    dataset="DEMO",
    column="mec_exam_weight",
    issue=(
        "1,376 cohort participants have MEC examination weight stored as the "
        "SAS XPT zero representation rather than numeric 0."
    ),
    example=(
        "1,376 mec_exam_weight values equal 5.397605346934028e-79. "
        "This matches the 1,376 participants without BMX/BPXO/GHB coverage."
    ),
    action_stage="cleaning",
    planned_action=(
        "Convert the XPT zero representation to numeric 0. During "
        "survey-weighted analyses involving MEC examination variables, "
        "use participants with an appropriate positive examination weight."
    )
)

add_audit_finding(
    dataset="GHB",
    column="phlebotomy_weight",
    issue=(
        "After merging to the cohort, 1,376 participants have no GHB component "
        "record and therefore have missing phlebotomy weight; an additional "
        "261 observed weight values are stored as the XPT zero representation."
    ),
    example=(
        "1,376 missing phlebotomy weights and 261 XPT-encoded zero weights."
    ),
    action_stage="cleaning",
    planned_action=(
        "Convert XPT-encoded zeros to numeric 0 and retain component "
        "noncoverage as missing. Use positive phlebotomy weights where "
        "appropriate for survey-weighted analyses."
    )
)

add_audit_finding(
    dataset="GLU",
    column="fasting_subsample_weight",
    issue=(
        "After merging to the cohort, 3,989 participants have no GLU component "
        "record and therefore have missing fasting subsample weight; an additional "
        "452 observed weights are stored as the XPT zero representation."
    ),
    example=(
        "3,989 missing fasting_subsample_weight values and "
        "452 XPT-encoded zero weights."
    ),
    action_stage="cleaning",
    planned_action=(
        "Convert XPT-encoded zeros to numeric 0 and retain component "
        "noncoverage as missing. Use positive fasting-subsample weights "
        "for analyses based on the fasting subsample."
    )
)

# 5. Saving Audit Logs

In [160]:
audit_findings = pd.DataFrame(audit_log)

audit_findings.to_csv(
    "../logs/audit_logs.csv",
    index=False
)

print("Saved to ../logs/audit_logs.csv")

Saved to ../logs/audit_logs.csv
